ETF Proxy Models - NAV, Risk and Hedging

In [ ]:
import pandas as pd
import numpy as np
import json
import sys
import warnings
from sklearn.linear_model import ElasticNetCV, RidgeCV

warnings.filterwarnings('ignore')

##### DO NOT MODIFY/DELETE THE BELOW CODE #############################################################################

#######################################################################################################################
# CONSTANTS STARTS
#######################################################################################################################

EOD_PRICES = pd.read_csv(
    "./eod_prices.csv",
    index_col=0,
    parse_dates=True
).sort_index()

TARGET_ETFS = [f"Target_ETF_{i}" for i in range(1, 6)]

# PROXY INSTRUMENTS AVAILABLE

# 1). ETFs
PROXY_ETFS = [f"Proxy_ETF_{i}" for i in range(1, 11)]

# 2). TREASURY FUTURES
TSY_FUTURES = ["TU", "FV", "TY", "US"]

# 3). TREASURY OTR BONDS
TSY_BONDS = ["UST_2Y", "UST_5Y", "UST_10Y", "UST_30Y"]

# 4). CREDIT DEFAULT SWAP INDICES
CDX = ["CDX_IG_5Y", "CDX_IG_10Y", "CDX_HY_5Y", "CDX_HY_10Y"]

# ALL PROXIES FOR NAV MODEL
ALL_PROXY = PROXY_ETFS + TSY_FUTURES + TSY_BONDS + CDX

# ONLY NON-ETF PROXIES FOR RISK MODEL
RISK_PROXY = TSY_FUTURES + TSY_BONDS + CDX

# PAR QUOTED INSTRUMENTS
PAR_QUOTED = set(TSY_FUTURES + TSY_BONDS + CDX)

# TRANSACTION COSTS
COST_BPS = {
    **{e: 2.0 for e in PROXY_ETFS},
    **{f: 0.5 for f in TSY_FUTURES},
    **{b: 0.5 for b in TSY_BONDS},
    **{c: 1.0 for c in CDX}
}

#######################################################################################################################
# CONSTANTS END
#######################################################################################################################

##### DO NOT MODIFY/DELETE THE BELOW CODE #############################################################################

#######################################################################################################################
# HELPER FUNCTIONS
#######################################################################################################################

def compute_returns(prices):
    """Compute simple daily returns from prices."""
    return prices.pct_change().dropna()

def predict_returns(weights, proxy_returns):
    """Predict target ETF returns from proxy returns and model weights."""
    return proxy_returns @ weights

def reconstruct_nav(weights, proxy_returns, base_nav):
    """Reconstruct NAV level from predicted returns chained off a base NAV."""
    pred_returns = predict_returns(weights, proxy_returns)
    return (1 + pred_returns).cumprod() * base_nav

def compute_mape(predicted_nav, actual_nav):
    """Compute MAPE."""
    aligned = pd.concat([predicted_nav, actual_nav], axis=1, join="inner").dropna()
    aligned.columns = ["predicted", "actual"]
    return (
        abs(aligned["predicted"] - aligned["actual"]) / aligned["actual"]
    ).mean() * 100

def compute_pnl_series(notionals: dict, prices: pd.DataFrame) -> pd.Series:
    """Compute PnL series."""
    if not notionals:
        return pd.Series(0.0, index=prices.index[1:])

    p0 = prices.iloc[0]

    scaled = pd.Series({
        c: n * (
            1.0 / p0[c] if c not in PAR_QUOTED else 1.0 / 100.0
        )
        for c, n in notionals.items()
    })

    return prices[scaled.index].diff().iloc[1:].mul(
        scaled,
        axis=1
    ).sum(axis=1)

def compute_cost(notionals: dict, prices: pd.DataFrame) -> float:
    """Compute transaction cost."""
    p0 = prices.iloc[0]

    return float(sum(
        abs(n) * COST_BPS[i] / 1e4
        if i not in PAR_QUOTED
        else (abs(n) / 100.0) * p0[i] * COST_BPS[i] / 1e4
        for i, n in notionals.items()
    ))

def compute_her(pnl_port: pd.Series, pnl_hedge: pd.Series) -> float:
    """Compute hedge effectiveness ratio."""
    var_p = pnl_port.var(ddof=0)

    return 0.0 if var_p <= 0 else float(
        1.0 - pnl_port.add(
            pnl_hedge,
            fill_value=0.0
        ).var(ddof=0) / var_p
    )

#######################################################################################################################
# YOUR CODE STARTS HERE
#######################################################################################################################

# -------------------------------------------------------------------
# COMPUTE RETURNS
# -------------------------------------------------------------------

returns = compute_returns(EOD_PRICES)

# -------------------------------------------------------------------
# PART 1 - NAV MODEL
# ElasticNetCV using ALL_PROXY
# -------------------------------------------------------------------

nav_model = pd.DataFrame(
    0.0,
    index=TARGET_ETFS,
    columns=ALL_PROXY
)

nav_model.index.name = "ETF"

X_nav = returns[ALL_PROXY].values

for etf in TARGET_ETFS:

    y = returns[etf].values

    model = ElasticNetCV(
        l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9],
        cv=5,
        fit_intercept=False,
        max_iter=20000,
        random_state=42
    )

    model.fit(X_nav, y)

    nav_model.loc[etf] = model.coef_

# -------------------------------------------------------------------
# PART 2 - RISK MODEL
# RidgeCV using only RISK_PROXY
# -------------------------------------------------------------------

risk_model = pd.DataFrame(
    0.0,
    index=TARGET_ETFS,
    columns=RISK_PROXY
)
risk_model.index.name = "ETF"

X_risk = returns[RISK_PROXY].values

alphas = np.logspace(-3, 5, 100)

for etf in TARGET_ETFS:

    y = returns[etf].values

    model = RidgeCV(
        alphas=alphas,
        cv=5,
        fit_intercept=False
    )

    model.fit(X_risk, y)

    risk_model.loc[etf] = model.coef_

# -------------------------------------------------------------------
# PART 3 - HEDGING
# -------------------------------------------------------------------

hedging_basket = pd.DataFrame(
    {"Notional": [0.0]},
    index=["TU"]
)

hedging_basket.index.name = "Instrument"

try:

    test_input = json.loads(sys.stdin.read())

    if test_input.get("key") == "hedge":

        portfolio = test_input.get("portfolio", {})

        hedge_notionals = {}

        p0 = EOD_PRICES.iloc[-1]

        for proxy in ALL_PROXY:

            exposure = 0.0

            for etf, notional in portfolio.items():

                if etf in nav_model.index:

                    w = nav_model.loc[etf, proxy]

                    if proxy in PAR_QUOTED:

                        scale = p0[proxy] / 100.0

                        exposure += notional * w / scale

                    else:

                        exposure += notional * w

            # Ignore tiny hedge positions
            if abs(exposure) > 1e-3:

                hedge_notionals[proxy] = -exposure

        if hedge_notionals:

            hedging_basket = pd.DataFrame(
                {"Notional": hedge_notionals}
            )

            hedging_basket.index.name = "Instrument"

except:
    pass

#######################################################################################################################
# YOUR CODE ENDS HERE
#######################################################################################################################

#######################################################################################################################
# SUBMISSION FORMAT
#######################################################################################################################

submission = {
    "nav": nav_model.to_csv(),
    "risk": risk_model.to_csv(),
    "hedge": hedging_basket.to_csv()
}

final_output = json.dumps(submission)

print(final_output)